## Configuração do Escopo de Banco de Dados

In [0]:
%sql
-- Camada Gold: Processamento e desnormalização cadastro
-- **Descrição:** Este script lê os dados transformado da camada Silver, realiza o processamento e desnormalização e salva os dados na camada Gold.
-- **Origem:** Silver
-- **Destino:** Gold
-- **Observações:** 
-- Criação Rodrigo Escarmeloto Franco

-- Definição do catálogo de trabalho
USE CATALOG portfolio_mercado_livre;

-- Criação automática do Schema Silver se não existir
CREATE SCHEMA IF NOT EXISTS gold;

## Criação e Carga da Dimensão Cliente

In [0]:
%sql
--  1. Tabela trusted: Clientes (trusted_clientes)

-- Criação da tabela Delta estruturada na Silver
CREATE OR REPLACE TABLE gold.dim_clientes (
    sk_cliente string
    ,id_cliente STRING
    ,nm_cliente STRING
    ,tipo_documento STRING
    ,estado_entrega STRING
    ,cidade_entrega STRING
    ,dh_insercao_gold TIMESTAMP
    ,dh_atualizacao_gold TIMESTAMP
    ,fl_ultima_versao SMALLINT
) USING DELTA;

-- Carga incremental/Sobrescrita da Dim Clientes
INSERT OVERWRITE gold.dim_clientes
select distinct
  sk_cliente
  ,id_cliente
  ,nickname_cliente as nm_cliente
  ,tipo_documento
  ,estado_entrega
  ,cidade_entrega
  ,dh_insercao_silver AS dh_insercao_gold
  ,dh_atualizacao_silver as dh_atualizacao_gold
  ,fl_ultima_versao
from silver.trusted_clientes
WHERE id_cliente IS NOT NULL;

-- Prévia dos dados normalizados de clientes e regiões
SELECT * FROM gold.dim_clientes LIMIT 5;

## Criação e Carga da Dimensão Produto

In [0]:
%sql
-- 2. Tabela trusted: Produtos (trusted_produtos)
CREATE OR REPLACE TABLE gold.dim_produtos (
    sk_produto STRING
    ,id_produto string
    ,preco_unitario DOUBLE
    ,quantidade_itens INT
    ,origem_dados STRING
    ,dh_ingestao_etl TIMESTAMP
    ,dh_insercao_gold TIMESTAMP
    ,dh_atualizacao_gold TIMESTAMP
    ,fl_ultima_versao SMALLINT
) USING DELTA;

-- Agrupando todos os produtos únicos da plataforma
INSERT OVERWRITE gold.dim_produtos
select 
  sk_produto
  ,id_produto
  ,preco_unitario
  ,quantidade_itens
  ,origem_dados
  ,dh_ingestao_etl
  ,dh_insercao_silver as dh_insercao_gold
  ,dh_atualizacao_silver as dh_atualizacao_gold
  ,fl_ultima_versao
from silver.trusted_produtos
WHERE id_produto IS NOT NULL;

SELECT * FROM gold.dim_produtos LIMIT 5;

## Criação e Carga da Dimensão Status Venda


In [0]:
%sql
-- 3. Tabela trusted: status venda (trusted_status_venda)
CREATE OR REPLACE TABLE gold.dim_status_venda (
    sk_status_venda STRING
    ,status_venda STRING
    ,dh_insercao_gold TIMESTAMP
    ,dh_atualizacao_gold TIMESTAMP
    ,fl_ultima_versao SMALLINT
) USING DELTA;

-- Agrupando todos os produtos únicos da plataforma
INSERT OVERWRITE gold.dim_status_venda
SELECT  
    sk_status_venda 
    ,status_venda
    ,dh_insercao_silver AS dh_insercao_gold
    ,dh_atualizacao_silver as dh_atualizacao_gold
    ,fl_ultima_versao
FROM silver.trusted_status_venda;

-- Prévia da tabela evento para auditoria de métricas
SELECT * FROM gold.dim_status_venda LIMIT 5;

## Criação e Carga da Dimensão calendario


In [0]:
%sql
-- 4. Tabela evento: Vendas (evento_venda)
CREATE OR REPLACE TABLE gold.dim_calendario (
     sk_calendario INT
    ,id_data DATE
    ,data_venda TIMESTAMP
    ,ano STRING
    ,mes STRING
    ,dia STRING
    ,dia_semana STRING
    ,trimestre STRING
    ,semestre STRING
) USING DELTA;

INSERT OVERWRITE gold.dim_calendario
SELECT 
     sk_calendario 
    ,id_data 
    ,data_venda 
    ,ano 
    ,mes 
    ,dia 
    ,dia_semana 
    ,trimestre 
    ,semestre 
FROM silver.trusted_calendario;

-- Prévia da tabela evento para auditoria de métricas
SELECT * FROM gold.dim_calendario LIMIT 5;

## Criação e Carga da fato vendas (Com Explode de Itens)

In [0]:
%sql
-- 5. Tabela evento: Vendas (evento_venda)
CREATE OR REPLACE TABLE gold.fato_vendas (
    id_venda STRING
    ,sk_calendario INT
    ,id_produto STRING
    ,sk_produto STRING
    ,id_cliente STRING
    ,sk_cliente STRING
    ,sk_status_venda STRING
    ,quantidade_itens INT
    ,valor_unitario DOUBLE
    ,valor_total_venda DOUBLE
    ,dh_insercao_gold TIMESTAMP
) USING DELTA;

INSERT OVERWRITE gold.fato_vendas
SELECT 
    id_venda
    ,sk_calendario
    ,id_produto
    ,sk_produto
    ,id_cliente
    ,sk_cliente
    ,sk_status_venda
    ,quantidade_itens
    ,valor_unitario
    ,valor_total_venda
    ,dh_insercao_silver AS dh_insercao_gold
FROM silver.evento_vendas;

-- Prévia da tabela evento para auditoria de métricas
SELECT * FROM gold.fato_vendas LIMIT 5;